# 10 — accepted M0 read-only audit

提示済みの M0 受理記録と `/storage` 上のチェックポイントを読み取り専用で再検証します。M0 を再実行・追記・再受理しません。成功しても最終 certification ではなく、M1 を開始できる親証拠の完全性だけを確認します。

In [1]:
# 必ず最初に pytest を用意する。
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('pytest') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pytest>=8'])
import pytest
print('pytest:', pytest.__version__)


pytest: 9.1.1


In [2]:
from pathlib import Path
import hashlib
import json
import os

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/storage') / BUNDLE_NAME,
]
PROJECT_ROOT = next((p.expanduser().resolve() for p in candidates
                     if (p.expanduser() / 'audit/m0_accepted_parent.json').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('audit/m0_accepted_parent.json を含む project root を検出できません。')
if sys.version_info < (3, 11):
    raise RuntimeError(f'Python 3.11+ が必要です: {sys.version.split()[0]}')
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)


PROJECT_ROOT = /notebooks/validated_4d_su2_rg_codex_bundle


In [3]:
def read_json(path: Path):
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

record_path = PROJECT_ROOT / 'audit/m0_accepted_parent.json'
record = read_json(record_path)
required_record = {
    'parent_milestone': 'M0',
    'accepted_phase': 'M0_COMPLETE',
    'certification_status': 'NOT_CERTIFIED',
    'restart_test_status': 'PASS',
}
for key, expected_value in required_record.items():
    if record.get(key) != expected_value:
        raise RuntimeError(f'M0 受理記録の {key} が不正です。')
if record.get('accepted_for_next_milestone') != 'M1':
    raise RuntimeError('M0 受理記録は M1 開始を許可していません。')

checkpoint = Path(record['parent_checkpoint_path']).expanduser().resolve()
if checkpoint.name != record['parent_checkpoint'] or checkpoint.is_symlink() or not checkpoint.is_dir():
    raise RuntimeError(f'M0 checkpoint が存在しないか安全でありません: {checkpoint}')
if any(path.is_symlink() for path in checkpoint.rglob('*')):
    raise RuntimeError('M0 checkpoint 内に symlink があります。')
for name in ('COMMITTED', 'hashes.json', 'state.json', 'work_queue.json'):
    if not (checkpoint / name).is_file():
        raise RuntimeError(f'M0 checkpoint に {name} がありません。')

manifest = read_json(checkpoint / 'hashes.json')
if not isinstance(manifest, dict) or not manifest:
    raise RuntimeError('hashes.json が空または不正です。')
for relative, expected_digest in manifest.items():
    if not isinstance(relative, str) or not isinstance(expected_digest, str) or len(expected_digest) != 64:
        raise RuntimeError('hashes.json の項目形式が不正です。')
    target = (checkpoint / relative).resolve()
    try:
        target.relative_to(checkpoint)
    except ValueError as exc:
        raise RuntimeError(f'hash 対象が checkpoint 外へ出ています: {relative}') from exc
    if not target.is_file() or sha256_file(target) != expected_digest:
        raise RuntimeError(f'M0 checkpoint hash mismatch: {relative}')
actual_files = {p.relative_to(checkpoint).as_posix() for p in checkpoint.rglob('*')
                if p.is_file() and p.name not in {'hashes.json', 'COMMITTED'}}
if actual_files != set(manifest):
    raise RuntimeError('M0 checkpoint のファイル集合が hashes.json と一致しません。')

state = read_json(checkpoint / 'state.json')
state_expected = {
    'run_id': record['parent_run_id'],
    'phase': 'M0_COMPLETE',
    'certification_status': 'NOT_CERTIFIED',
    'checkpoint_index': record['checkpoint_index'],
}
for key, expected_value in state_expected.items():
    if state.get(key) != expected_value:
        raise RuntimeError(f'M0 state.json の {key} が受理記録と一致しません。')

queue_payload = read_json(checkpoint / 'work_queue.json')
raw_items = queue_payload.get('items') if isinstance(queue_payload, dict) else None
if isinstance(raw_items, dict):
    keyed_items = list(raw_items.items())
elif isinstance(raw_items, list):
    keyed_items = [(str(index), item) for index, item in enumerate(raw_items)]
else:
    raise RuntimeError('work_queue.json の items が不正です。')
items = []
for fallback_id, item in keyed_items:
    if not isinstance(item, dict):
        raise RuntimeError('work queue item が object ではありません。')
    normalized = dict(item)
    normalized.setdefault('item_id', fallback_id)
    items.append(normalized)
statuses = ('done', 'pending', 'running', 'failed')
counts = {status: sum(item.get('status') == status for item in items) for status in statuses}
if counts != record['queue']:
    raise RuntimeError(f'M0 queue counts mismatch: {counts}')

run_root = checkpoint.parents[1].resolve()
for item in items:
    if item.get('status') != 'done':
        raise RuntimeError(f'未完了 M0 item があります: {item.get("item_id")}')
    item_id = item['item_id']
    relative = item.get('result_relpath')
    expected_digest = item.get('result_sha256')
    if not relative or not expected_digest:
        raise RuntimeError(f'M0 done item に result metadata がありません: {item_id}')
    result_path = (run_root / relative).resolve()
    try:
        result_path.relative_to(run_root)
    except ValueError as exc:
        raise RuntimeError(f'result path が run root 外へ出ています: {item_id}') from exc
    if not result_path.is_file() or sha256_file(result_path) != expected_digest:
        raise RuntimeError(f'M0 result artifact が欠損または破損しています: {item_id}')
    marker_path = run_root / 'work_items' / f'{item_id}.done'
    marker = read_json(marker_path) if marker_path.is_file() else None
    if not isinstance(marker, dict) or any((
        marker.get('item_id') != item_id,
        marker.get('result_relpath') != relative,
        marker.get('result_sha256') != expected_digest,
    )):
        raise RuntimeError(f'M0 done marker が欠損または不整合です: {item_id}')

M0_AUDIT = {
    'status': 'PASS',
    'milestone': 'M0',
    'phase': 'M0_COMPLETE',
    'certification_status': 'NOT_CERTIFIED',
    'independent_checkpoint_audit_performed_now': True,
    'checkpoint': str(checkpoint),
    'checkpoint_manifest_sha256': sha256_file(checkpoint / 'hashes.json'),
    'acceptance_record_sha256': sha256_file(record_path),
    'queue': counts,
}
M0_AUDIT


{'status': 'PASS',
 'milestone': 'M0',
 'phase': 'M0_COMPLETE',
 'certification_status': 'NOT_CERTIFIED',
 'independent_checkpoint_audit_performed_now': True,
 'checkpoint': '/storage/validated_4d_su2_rg/runs/20260719T120406Z-731966c8fd28/checkpoints/ckpt_000014',
 'checkpoint_manifest_sha256': '9caa0e97593b21b3654221f77f3b86dc7edb84c980cf1c0fda02e85c50ad6c79',
 'acceptance_record_sha256': 'fce23bda6ba04fc69fff59624356a52b291bee5b273f0b57e4ec5975027c5ecb',
 'queue': {'done': 6, 'pending': 0, 'running': 0, 'failed': 0}}

## 判定

上のセルが `PASS` を返した場合に限り、fresh kernel で `20_m1_exact_2d.ipynb` へ進みます。この監査は `audit/m0_accepted_parent.json` や受理済み M0 checkpoint を一切変更しません。